In [1]:
import os
import pickle
import pandas as pd
import numpy as np
from datetime import datetime

In [2]:
%pip install xgboost


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install pyarrow


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
## set up configuration

In [5]:
config = {
    "model_bank_directory": "model_bank",
    "model_name": "champion_model.pkl",
    "feature_store_path": "datamart/gold/feature_store",
    "inference_start_date": "2024-07-01",
    "inference_end_date": "2024-12-01",
    "prediction_output_directory": "datamart/gold/model_predictions/champion_model"
}

config["model_artefact_filepath"] = os.path.join(
    config["model_bank_directory"],
    config["model_name"]
)

config


{'model_bank_directory': 'model_bank',
 'model_name': 'champion_model.pkl',
 'feature_store_path': 'datamart/gold/feature_store',
 'inference_start_date': '2024-07-01',
 'inference_end_date': '2024-12-01',
 'prediction_output_directory': 'datamart/gold/model_predictions/champion_model',
 'model_artefact_filepath': 'model_bank/champion_model.pkl'}

In [6]:
# =========================================================
# 2. Load champion model artefact
# =========================================================

with open(config["model_artefact_filepath"], "rb") as file:
    model_artefact = pickle.load(file)

model = model_artefact["model"]
feature_cols = model_artefact["feature_cols"]

print("Loaded champion model:", model_artefact["model_name"])
print("Model version:", model_artefact["model_version"])
print("Number of features:", len(feature_cols))


Loaded champion model: xgboost
Model version: credit_model_xgboost_2026_06_17
Number of features: 43


## Load Gold Feature Store

In [7]:
# =========================================================
# 3. Load inference data
# =========================================================
# Use your merged table or feature store.
# Inference rows are rows where label is NaN.

import pandas as pd

months = [
    "2024_07_01",
    "2024_08_01",
    "2024_09_01",
    "2024_10_01",
    "2024_11_01",
    "2024_12_01"
]

dfs = []

for month in months:

    filepath = (
        "datamart/gold/feature_store/"
        f"gold_feature_store_{month}.parquet"
    )

    df = pd.read_parquet(filepath)

    dfs.append(df)

inference_df = pd.concat(
    dfs,
    ignore_index=True
)

print(inference_df.shape)

(3000, 52)


In [8]:
# =========================================================
# 4. Prepare X inference
# =========================================================

X_inference = inference_df[feature_cols].copy()

X_inference = X_inference.replace([np.inf, -np.inf], np.nan)
X_inference = X_inference.fillna(X_inference.median(numeric_only=True))


In [9]:
# =========================================================
# 5. Make predictions
# =========================================================

pred_proba = model.predict_proba(X_inference)[:, 1]
pred_label = model.predict(X_inference)

prediction_df = inference_df[
    ["Customer_ID", "snapshot_date"]
].copy()

prediction_df["model_name"] = model_artefact["model_name"]
prediction_df["model_version"] = model_artefact["model_version"]
prediction_df["pred_proba"] = pred_proba
prediction_df["pred_label"] = pred_label

prediction_df.head()

,Customer_ID,snapshot_date,model_name,model_version,pred_proba,pred_label
0,cus_0x1108,2024-07-01,xgboost,credit_model_xgboost_2026_06_17,0.123586,0
1,cus_0x13b5,2024-07-01,xgboost,credit_model_xgboost_2026_06_17,0.114135,0
2,cus_0x13b8,2024-07-01,xgboost,credit_model_xgboost_2026_06_17,0.108544,0
3,cus_0x144f,2024-07-01,xgboost,credit_model_xgboost_2026_06_17,0.564501,1
4,cus_0x14ec,2024-07-01,xgboost,credit_model_xgboost_2026_06_17,0.182708,0


In [11]:
# =========================================================
# 6. Save predictions as gold table
# =========================================================

os.makedirs(config["prediction_output_directory"], exist_ok=True)

output_path = (
    config["prediction_output_directory"]
    + "_predictions.parquet"
)

prediction_df.to_parquet(output_path, index=False)

print("Saved predictions to:", output_path)
print(prediction_df.shape)

Saved predictions to: datamart/gold/model_predictions/champion_model_predictions.parquet
(3000, 6)
